In [1]:
from tqdm import tqdm
import pickle
import glob
from tqdm import tqdm
import pickle
import glob
from experiments.end_to_end.proof_node import ErrorNode, Status
import math


bestfs_path = "../runs/internlm/internlm/2025_02_14/16_25_13/traces/0/"
cg_path = "../runs/internlm/internlm_cg/2025_02_18/12_00_59/traces/0/"
dpp_path = "../runs/internlm/dpp_critic/2025_02_24/16_13_55/traces/0/"
dpp_path_2 = "../runs/internlm/dpp_critic/2025_03_08/02_33_49"
def load_traces(path):
    files = glob.glob(path + '*', recursive=True)
    traces = []

    for file in tqdm(files):
        traces.append(pickle.load(open(file, "rb")))

    return traces

In [ ]:
# bfs_traces = load_traces(bfs_path)
bestfs_traces = load_traces(bestfs_path)
cg_traces = load_traces(cg_path)

In [2]:
dpp_traces = load_traces(dpp_path)

100%|██████████| 236/236 [00:05<00:00, 45.21it/s]


In [ ]:
best_fs_proved = [a.theorem.full_name for a in bestfs_traces if a.proof]
critic_proved = [a.theorem.full_name for a in cg_traces if a.proof]

In [ ]:
len(best_fs_proved), len(critic_proved)

In [ ]:
# get the union ov proven diversity theorems
all_proved = set( best_fs_proved + critic_proved )#+ diversity_p99_t1_proved) #+ diversity_p99_t2_proved)
len(all_proved) / 245

In [ ]:
# get the intersection of proved theorems, and number unique to each set

best_fs_proved_set = set(best_fs_proved)

intersection = diversity_proved.intersection(best_fs_proved_set)

diversity_unique = diversity_proved.difference(best_fs_proved_set)

best_fs_unique = best_fs_proved_set.difference(diversity_proved)


len(intersection), len(diversity_unique), len(best_fs_unique)

In [ ]:

len([t for t in cg_traces if t.env_time >= 600])

In [ ]:
cg_traces[0].num_expansions


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.hist([len(t.proof) for t in cg_traces if t.proof])

In [ ]:

plt.hist([len(t.proof) for t in bestfs_traces if t.proof])


In [ ]:
32 * 600

In [ ]:
plt.hist([t.num_expansions for t in cg_traces if not t.proof])


In [ ]:

plt.hist([t.num_expansions for t in bestfs_traces        if not t.proof])


In [ ]:
plt.hist([t.env_time for t in cg_traces], bins=range(0, 700, 10))


In [ ]:
tac_times = [a.time for trace in bestfs_traces for x in trace.data['search_trace'] for a in x]


In [ ]:
# plot histogram of times with seaborn
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(tac_times,)
plt.xlim(0, 2)
plt.show()



In [ ]:
[(a.tactic, a.time) for x in bestfs_traces[0].data['search_trace'] for a in x]


In [ ]:
[(a.dst[0].inner.message.split(' tactic_state')[0]) for x in bestfs_traces[1].data['search_trace'] for a in x if isinstance(a.dst[0], ErrorNode)]


In [ ]:
# train data path
train_path = 'runs/train_traces/0/'

In [ ]:
train_files = glob.glob(train_path + '*', recursive=True)

In [ ]:
from experiments.end_to_end.proof_node import Status

In [ ]:
# takeaway: fails distributed zipfian, with a few proofs having most fails
# fails with no out_edges are usually root nodes which have errored out
# fails with out_edges are usually nodes which haven't been fully visited, implying an error in one of the expansions

In [ ]:
weird_fails = []
fail_dist = []
true_fails = 0
no_edges = []
for file in tqdm(train_files):
    trace = pickle.load(open(file, 'rb'))

    fail_nodes = [node for node in trace.nodes.values() if node.status == Status.FAILED]
    for node in fail_nodes:
        if node.out_edges:
            if all([any([c.status == Status.FAILED for c in child.dst]) for child in
                    node.out_edges]) and node.visit_count >= node.max_expansions:
                true_fails += 1
            else:
                weird_fails.append(node)
        else:
            no_edges.append(node)

    fail_dist.append(len(fail_nodes))


In [ ]:
len(weird_fails), true_fails, len(no_edges)

In [ ]:
[a.visit_count for a in weird_fails]

In [ ]:
[(type(d.dst[0]), len(d.dst)) for d in weird_fails[0].out_edges]


In [ ]:
weird_fails[0].in_edges

In [ ]:
weird_fails[0]

In [ ]:
# plot histogram of fail_dist, excluding those with 0 value

import matplotlib.pyplot as plt

plt.hist(fail_dist, bins=range(1, max(fail_dist) + 1))
plt.show()


In [ ]:
[type(d.dst[0]) for d in [node for node in trace.nodes.values() if node.status == Status.FAILED][1].out_edges]
# [d.dst[0] for d in [node for node in trace.nodes.values() if node.status == Status.FAILED][0].out_edges]


In [ ]:
# look at distribution of visits for proven nodes

proven_nodes = []
other_nodes = []

visits = {}

for trace in tqdm(bestfs_traces):
    if isinstance(trace.tree, ErrorNode) or not trace.tree.out_edges:
        continue

    nodes = trace.nodes
    nodes[trace.tree.goal] = trace.tree

    visits = {node: nodes[node].visit_count for node in nodes.keys()}

    for goal, node in nodes.items():
        for a in node.ancestors:
            visits[a] += node.visit_count

    for node in trace.nodes.values():
        if node.status == Status.PROVED:
            proven_nodes.append((node, visits[node.goal]))
        else:
            other_nodes.append((node, visits[node.goal]))


In [ ]:
len(proven_nodes)

In [ ]:
len([node for node in other_nodes if node[0].status == Status.FAILED])

In [ ]:
# plot normalised distribution of visits for proven nodes

import matplotlib.pyplot as plt

plt.hist([d[1] for d in proven_nodes], bins=range(0, 4000, 64))


In [ ]:
# print the number of proven nodes with visit counts over 64, 128, 256, 512, 1024, 2048, 4096, 8192

for i in range(8):
    num_over = len([d[0] for d in proven_nodes if d[1] > 64 * 2 ** i]) / len(proven_nodes)
    print(f'Num over {64 * 2 ** i}: {num_over}')

In [ ]:

for i in range(8):
    num_over = len([d[0] for d in other_nodes if d[1] >= 64 * 2 ** i]) / len(other_nodes)
    print(f'Num over {64 * 2 ** i}: {num_over}')


In [ ]:
htps_path = 'runs/end_to_end/htps_no_critic/2024_04_21/16_34_02/traces/0/'

htps_files = glob.glob(htps_path + '*', recursive=True)

In [ ]:
errors = 0
for file in tqdm(htps_files):
    trace = pickle.load(open(file, 'rb'))
    if trace.tree.status == Status.FAILED:
        errors += 1


In [ ]:
errors

In [ ]:
trace = pickle.load(open(htps_files[13], 'rb'))

print(len(trace.data['search_trace']))
print(trace.tree.status)

edge_data, tree, leaves = trace.data['search_trace'][-1]

edge_data